# QMS Wait-Time ML Pipeline

**Purpose**: Train and validate a HistGradientBoosting model to predict queue waiting time for Malaysian bank branches.

**Pipeline Lifecycle**:
1. Data Ingestion (synthetic + optional external datasets)
2. Data Cleansing (abandoned tickets, outliers, dedup)
3. Feature Engineering (26 features aligned with production)
4. Model Training (temporal split, sample weighting, quantile p75 loss)
5. Evaluation & Comparison (ML vs Formula baseline)

**System**: QMS (Queue Management System) — online booking + walk-in, multi-counter, CreatedAt FIFO call ordering

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

print('Libraries loaded.')

## 1. Data Ingestion

Generate synthetic data that matches production system behavior (the simulation cells below are copied verbatim from `ml/generate_seed_data.py`; edit both together):
- **Production services**: codes, names, ticket prefixes and default minutes from `database/insert-service-types-all-branches.sql` + `insert-service-queues-all-branches.sql` (1 service = 1 queue = 1 prefix: A = Account Opening, D = Deposit, W = Withdraw, L = Loan, C = Credit Card, F = Fixed Deposit, I = Investment, B = Bancassurance, T = ATM/CDM, O = Online Banking, S = Business Banking, R = Remittance, Q = Customer Service)
- **Production branches**: real BranchCodes, 60-min slots aligned to opening time, Mon-Fri 09:30-16:00, `OnlineSlotsPerSlot` (manager setting per service, `ServiceType.OnlineSlotsPerSlot`) drawn per branch-day per service from 2..8; bookings beyond that capacity become walk-ins, and the emitted feature equals the capacity used
- **CreatedAt FIFO** call ordering (matches `QueueCallNextSelector`)
- **Multi-counter** drain with service assignments (matches `CounterSimulationEstimator`)
- **People Ahead (per counter)**: for each counter that serves my lane, `PeopleAheadCallNext` counts eligible, not-yet-called tickets in THAT counter's lanes that Call Next picks before me by (CreatedAt, Id), and takes the minimum over those counters (same-lane only if no counter serves my lane); `CrossLaneQueueLength` stays the waiting count over the union S of those lanes, so People Ahead <= CrossLane. Same definition as live `CallNextQueueMath` and the customer People Ahead
- **26 features** (incl. `EarlyCallMinutes` drawn per branch-day from {0,5,10,15,20,30}) aligned with `MlHttpWaitTimeEstimator.BuildPayload` / `WaitTimeFeatureBuilder`
- **EnqueueSequence** = per branch-day rank by CreatedAt across all services (same as live)
- **Check-in time** for online bookings: 80% arrive 0-20 min early, 20% arrive 0-15 min late; `QueueEligibleAt = max(CheckInAt, SlotStart - EarlyCall)`
- **Target** = `CalledAt - QueueEligibleAt`, where CalledAt is the moment a counter picks the ticket (matches `MlTrainingObservation.ActualWaitingMinutes`; never negative)

In [ ]:
# ── Branch & Service configs (matching the production seed scripts in database/) ──

# Real production BranchCodes (insert-branches-from-paste.sql: BranchCode = 101000 + row).
# Production branch settings: SlotDurationMinutes = 60, OnlineEarlyCallMinutes = 10 (the simulator
# draws early call per branch-day from EARLY_CALL_CHOICES instead), Mon-Fri 09:30-16:00.
BRANCHES = [
    {"BranchCode": 101033, "name": "Bandar Sri Permaisuri", "base_traffic": 1.0, "slot_duration": 60, "early_call": 10},
    {"BranchCode": 101038, "name": "Taman Midah",           "base_traffic": 0.8, "slot_duration": 60, "early_call": 10},
    {"BranchCode": 101008, "name": "Bangsar",               "base_traffic": 1.3, "slot_duration": 60, "early_call": 10},
    {"BranchCode": 101039, "name": "Taman Connaught",       "base_traffic": 0.9, "slot_duration": 60, "early_call": 10},
    {"BranchCode": 101036, "name": "Wangsa Maju",           "base_traffic": 1.1, "slot_duration": 60, "early_call": 10},
]

# Branch service window (BRANCH_OPERATING_HOURS seed / BranchOperatingHoursBackfill): 09:30-16:00.
# Slots are aligned to opening time (QmsQueueService.AlignSlot anchor), so 60-min slots are
# 09:30, 10:30, ... 14:30; a slot must end by closing time, so 14:30-15:30 is the last bookable slot.
OPEN_MIN = 9 * 60 + 30    # 570
CLOSE_MIN = 16 * 60       # 960
DRAIN_END_MIN = CLOSE_MIN + 90  # counters keep calling the remaining queue until 17:30

# Production services: code, name, prefix (SERVICE_QUEUES.TicketPrefix) and DefaultAvgServiceMinutes.
# "weight" = share of arrivals (sim only). OnlineSlotsPerSlot is not fixed here: it is a manager setting
# per service (ServiceType.OnlineSlotsPerSlot), so it is drawn per branch-day below.
SERVICES = [
    {"code": "ACC_OPEN", "name": "Account Opening",           "prefix": "A", "avg_min": 15, "weight": 0.08},
    {"code": "DEPOSIT",  "name": "Deposit Services",          "prefix": "D", "avg_min": 3,  "weight": 0.16},
    {"code": "WITHDRAW", "name": "Withdrawal Services",       "prefix": "W", "avg_min": 4,  "weight": 0.06},
    {"code": "LOAN",     "name": "Loan / Financing",          "prefix": "L", "avg_min": 30, "weight": 0.08},
    {"code": "CCARD",    "name": "Credit Card Services",      "prefix": "C", "avg_min": 10, "weight": 0.10},
    {"code": "FDEP",     "name": "Fixed Deposit",             "prefix": "F", "avg_min": 15, "weight": 0.06},
    {"code": "INVUT",    "name": "Investment / Unit Trust",   "prefix": "I", "avg_min": 30, "weight": 0.04},
    {"code": "BANCA",    "name": "Insurance / Bancassurance", "prefix": "B", "avg_min": 25, "weight": 0.06},
    {"code": "ATMCDM",   "name": "ATM / CDM Services",        "prefix": "T", "avg_min": 5,  "weight": 0.10},
    {"code": "ONLBANK",  "name": "Online Banking Support",    "prefix": "O", "avg_min": 5,  "weight": 0.05},
    {"code": "BIZBANK",  "name": "Business Banking",          "prefix": "S", "avg_min": 20, "weight": 0.06},
    {"code": "REMIT",    "name": "Remittance / Transfer",     "prefix": "R", "avg_min": 15, "weight": 0.05},
    {"code": "CUSTSVC",  "name": "Queue / Customer Service",  "prefix": "Q", "avg_min": 10, "weight": 0.14},
]

# Counter profiles: which prefixes (= services) a counter listens to (COUNTER_ALLOWED_SERVICES).
# The production seed creates counters with no allowed services (the manager assigns them), so these
# are realistic teller groupings; every prefix is covered by at least one profile.
PROFILES = [
    ["D", "W", "Q"],            # cash teller: deposit, withdraw, customer service
    ["D", "W", "T"],            # cash teller: deposit, withdraw, ATM/CDM
    ["L", "C"],                 # credit: loan, credit card
    ["A", "F", "I", "B"],       # sales: account opening, fixed deposit, investment, bancassurance
    ["S", "O", "R"],            # business banking, online banking, remittance
    ["D", "W", "L", "C", "Q"],  # multi-skill
]

# Realistic manager settings for Branch.OnlineEarlyCallMinutes (drawn per branch-day)
EARLY_CALL_CHOICES = [0, 5, 10, 15, 20, 30]

# Manager setting ServiceType.OnlineSlotsPerSlot (max online bookings per slot for one service; the same
# value applies to every slot of that service). Drawn per branch-day per service, uniform integer 2..8.
ONLINE_SLOTS_MIN = 2
ONLINE_SLOTS_MAX = 8

print(f'{len(BRANCHES)} branches, {len(SERVICES)} services, {len(PROFILES)} counter profiles')

In [ ]:
from __future__ import annotations  # PEP 604 annotations on Python < 3.10
from dataclasses import dataclass

@dataclass
class Ticket:
    """A simulated queue ticket."""
    ticket_id: int
    branch_code: int
    service_code: str
    prefix: str
    default_avg_min: float
    online_slots_per_slot: int
    entry_type: int  # 0=OnlineBooked, 1=WalkIn
    checked_in: bool
    created_at: float  # minutes from midnight (online = booking time, may be hours earlier)
    slot_start: float | None = None  # for online bookings
    slot_end: float | None = None
    queue_eligible_at: float = 0.0  # walk-in=created_at, online=max(checkin_at, slot_start - early_call)
    serving_started_at: float | None = None  # = CalledAt: the moment a counter picks the ticket
    service_end_at: float | None = None  # actual end time (observable: counter becomes free)
    abandoned: bool = False
    enqueue_seq: int = 0  # per branch-day rank by (created_at, ticket_id), across all services
    checkin_at: float | None = None  # physical check-in time (online) / created_at (walk-in)


@dataclass
class Counter:
    """A simulated service counter."""
    counter_id: int
    allowed_prefixes: list[str]
    free_at: float = 0.0  # when this counter becomes free
    currently_serving: Ticket | None = None


def hour_traffic(hour: int) -> float:
    return {9: 1.4, 10: 1.6, 11: 1.3, 12: 0.7, 13: 0.8, 14: 1.3, 15: 1.5, 16: 1.2}.get(hour, 0.5)


def day_traffic(dow: int) -> float:
    # 0=Mon..4=Fri, 5=Sat, 6=Sun (python weekday; live WaitTimeFeatureBuilder uses the same encoding)
    return {0: 1.4, 1: 1.1, 2: 1.0, 3: 1.0, 4: 1.3}.get(dow, 0.0)

print('Simulation classes defined.')

In [ ]:
from __future__ import annotations  # PEP 604 annotations on Python < 3.10

def heard_lanes(prefix: str, counters: list[Counter]) -> tuple[list[set[str]], set[str]]:
    """Listening counters' prefix sets and the heard lane set S (live CallNextQueueMath.ResolveHeardLanesAsync).

    A counter listens to my prefix if its allowed_prefixes contain it (an empty list serves nothing).
    S = union of the listening counters' prefixes (includes mine); S = {my prefix} if no counter serves it.
    """
    listening = [set(c.allowed_prefixes) for c in counters if c.allowed_prefixes and prefix in c.allowed_prefixes]
    heard = set().union(*listening) if listening else {prefix}
    return listening, heard


def people_ahead_call_next(t: Ticket, tickets: list[Ticket], counters: list[Counter], snap_time: float) -> int:
    """PeopleAheadCallNext of ticket t at snap_time (mirrors live CallNextQueueMath.CountPeopleAheadAsync).

    Eligible ahead of me = not abandoned, checked in, queue_eligible_at <= snap_time, not yet picked
    (serving_started_at None or > snap_time) and before me by the pick key (created_at, ticket_id), which
    equals the live QueueCallNextSelector key (CreatedAt, Id). I am never counted.
    For each listening counter, count the eligible-ahead tickets in THAT counter's prefixes; return the
    minimum over the counters. If no counter serves my prefix, count same-prefix tickets only.
    Example: counter1 {A,B}, counter2 {B}; A001, A002, A003, B001 -> B001 = min(3, 0) = 0.
    """
    listening, heard = heard_lanes(t.prefix, counters)
    my_key = (t.created_at, t.ticket_id)
    ahead_prefixes = [
        o.prefix for o in tickets
        if o is not t
        and o.prefix in heard
        and not o.abandoned
        and o.checked_in
        and o.queue_eligible_at <= snap_time
        and (o.serving_started_at is None or o.serving_started_at > snap_time)
        and (o.created_at, o.ticket_id) < my_key
    ]
    if not listening:
        return sum(1 for p in ahead_prefixes if p == t.prefix)
    return min(sum(1 for p in ahead_prefixes if p in lanes) for lanes in listening)


def simulate_day(
    rng: np.random.Generator,
    branch: dict,
    day_of_week: int,
    n_profiles: int,
) -> list[dict]:
    """Simulate one full day at one branch using discrete-event Call Next drain."""

    svc_codes = [s["code"] for s in SERVICES]
    svc_weights = np.array([s["weight"] for s in SERVICES], dtype=float)
    svc_weights /= svc_weights.sum()
    svc_by_code = {s["code"]: s for s in SERVICES}

    slot_dur = branch["slot_duration"]
    # Last bookable slot must end by closing time (QmsQueueService rejects slotEnd > window end)
    last_slot_idx = (CLOSE_MIN - OPEN_MIN) // slot_dur - 1
    # Manager-configurable Branch.OnlineEarlyCallMinutes: draw per branch-day so the
    # model sees the full realistic range (0..30) instead of a fixed per-branch value.
    early_call = int(rng.choice(EARLY_CALL_CHOICES))
    # Manager-configurable ServiceType.OnlineSlotsPerSlot: one value per service for this branch-day.
    online_cap = {
        s["code"]: int(rng.integers(ONLINE_SLOTS_MIN, ONLINE_SLOTS_MAX + 1))
        for s in SERVICES
    }

    # Set up counters
    profile_indices = rng.choice(len(PROFILES), size=n_profiles, replace=False)
    counters = [
        Counter(counter_id=i, allowed_prefixes=PROFILES[idx])
        for i, idx in enumerate(profile_indices)
    ]

    day_mult = day_traffic(day_of_week)
    if day_mult == 0:
        return []

    tickets: list[Ticket] = []
    ticket_id = 0
    online_used: dict[tuple[str, int], int] = {}  # (service_code, slot_idx) -> online bookings

    # Seeded before arrivals so the RNG draw order matches the notebook.
    # Rolling avg service times per prefix (each production service has its own prefix/queue).
    # Each entry is (completion_time, duration) so snapshots can filter by time.
    # Seed with 10 baseline durations per prefix (time=0 = before day starts).
    completed_durations: dict[str, list[tuple[float, float]]] = {
        s["prefix"]: [(0.0, s["avg_min"] * rng.uniform(0.6, 1.4)) for _ in range(10)]
        for s in SERVICES
    }

    # Generate arrivals inside the service window 09:30-16:00 (first/last hour are partial)
    for hour in range(OPEN_MIN // 60, (CLOSE_MIN + 59) // 60):
        lo = max(hour * 60, OPEN_MIN)
        hi = min(hour * 60 + 60, CLOSE_MIN)
        if hi <= lo:
            continue
        h_mult = hour_traffic(hour)
        lam = branch["base_traffic"] * h_mult * day_mult * 8 * (hi - lo) / 60  # ~8 per base hour
        n_arrivals = max(0, rng.poisson(lam))

        for _ in range(n_arrivals):
            created_at = rng.uniform(lo, hi)

            svc_code = str(rng.choice(svc_codes, p=svc_weights))
            svc = svc_by_code[svc_code]

            # 35% try to book online, 65% walk-in
            entry_type = 0 if rng.random() < 0.35 else 1

            slot_start = None
            slot_end = None
            queue_eligible_at = created_at
            checkin_at = created_at  # walk-in: check-in = ticket creation

            if entry_type == 0:
                # Slot aligned to opening time; clamp to the last bookable slot
                slot_idx = min(int((created_at - OPEN_MIN) // slot_dur), last_slot_idx)
                # Online capacity: OnlineSlotsPerSlot per service per slot; when full, the customer walks in
                if online_used.get((svc_code, slot_idx), 0) >= online_cap[svc_code]:
                    entry_type = 1
                else:
                    online_used[(svc_code, slot_idx)] = online_used.get((svc_code, slot_idx), 0) + 1

            if entry_type == 0:  # OnlineBooked
                # Online booking: created_at = booking time (hours earlier) -> CreatedAt FIFO priority
                booking_hours_before = rng.exponential(4)
                actual_created = created_at - booking_hours_before * 60
                slot_start = OPEN_MIN + slot_idx * slot_dur
                slot_end = slot_start + slot_dur
                created_at = actual_created  # booking time

                checked_in = rng.random() < 0.88
                # ~8% abandon (never check in or leave before called)
                abandoned = not checked_in and rng.random() < 0.6
                if checked_in:
                    # Arrival relative to slot start: 80% early (0-20 min), 20% late (0-15 min)
                    offset = -rng.uniform(0, 20) if rng.random() < 0.8 else rng.uniform(0, 15)
                    checkin_at = slot_start + offset
                    # Eligible once physically checked in AND early-call window has opened
                    queue_eligible_at = max(checkin_at, slot_start - early_call)
                else:
                    checkin_at = None
                    queue_eligible_at = slot_start  # never served; dropped in cleaning
            else:  # WalkIn
                checked_in = True
                abandoned = rng.random() < 0.05  # 5% walk-ins abandon

            ticket_id += 1
            tickets.append(Ticket(
                ticket_id=ticket_id,
                branch_code=branch["BranchCode"],
                service_code=svc_code,
                prefix=svc["prefix"],
                default_avg_min=svc["avg_min"],
                online_slots_per_slot=online_cap[svc_code],  # = capacity actually enforced above
                entry_type=entry_type,
                checked_in=checked_in,
                created_at=created_at,
                slot_start=slot_start,
                slot_end=slot_end,
                queue_eligible_at=queue_eligible_at,
                abandoned=abandoned,
                enqueue_seq=0,
                checkin_at=checkin_at,
            ))

    if not tickets:
        return []

    # EnqueueSequence: per branch-day across all services, ranked by CreatedAt (booking time for online,
    # take-number time for walk-in), ticket_id tiebreak. Same definition as live WaitTimeFeatureBuilder.
    for rank, t in enumerate(sorted(tickets, key=lambda x: (x.created_at, x.ticket_id)), start=1):
        t.enqueue_seq = rank

    # ── Discrete-event Call Next drain ──
    # Simulate counters pulling tickets in CreatedAt FIFO order (matching QueueCallNextSelector)
    waiting = [t for t in tickets if not t.abandoned]
    # For online: only eligible when checkin_at <= now and slot_start - early_call <= now
    # For walk-in: eligible when checked_in

    def rolling_avg(prefix: str, before: float | None = None) -> float:
        """Rolling avg of last 30 completed durations, optionally filtered to completions before a given time."""
        entries = completed_durations[prefix]
        if before is not None:
            entries = [(t, d) for t, d in entries if t <= before]
        durations = [d for _, d in entries[-30:]]
        return float(np.mean(durations)) if durations else 10.0

    def is_eligible(t: Ticket, now: float) -> bool:
        if t.serving_started_at is not None or t.abandoned:
            return False
        if t.entry_type == 1:  # walk-in
            # Walk-in only exists after they arrive (created_at = arrival time)
            return t.checked_in and t.created_at <= now
        # online
        if not t.checked_in or t.slot_start is None or t.checkin_at is None:
            return False
        # Must be physically checked in and within the early-call window
        return t.checkin_at <= now and (t.slot_start - early_call) <= now

    def get_eligible_pool(now: float) -> list[Ticket]:
        return [t for t in waiting if is_eligible(t, now) and t.serving_started_at is None]

    # Event-driven: find next counter free time, pull ticket
    current_time = float(OPEN_MIN)
    end_time = float(DRAIN_END_MIN)

    for c in counters:
        c.free_at = current_time

    guard = 0
    max_iterations = len(waiting) * 3 + 1000

    while guard < max_iterations:
        guard += 1

        # Find earliest free counter
        earliest_counter = min(counters, key=lambda c: c.free_at)
        now = earliest_counter.free_at

        if now > end_time:
            break

        # Find eligible tickets for this counter's prefixes
        pool = [t for t in get_eligible_pool(now)
                if t.prefix in earliest_counter.allowed_prefixes]

        if not pool:
            # No eligible tickets — advance time to the next moment something can become eligible
            next_eligible_time = end_time
            for t in waiting:
                if t.serving_started_at is not None or t.abandoned:
                    continue
                if t.entry_type == 0 and t.checked_in and t.slot_start is not None:
                    eligible_at = max(t.checkin_at, t.slot_start - early_call)
                    if eligible_at > now:
                        next_eligible_time = min(next_eligible_time, eligible_at)

            if next_eligible_time >= end_time:
                # Check if any other counter still has work
                all_free = all(c.free_at <= now for c in counters)
                if all_free:
                    break
                else:
                    earliest_counter.free_at = now + 1  # advance 1 min
                    continue

            earliest_counter.free_at = next_eligible_time
            continue

        # CreatedAt FIFO: pick earliest CreatedAt (matching QueueCallNextSelector)
        pool.sort(key=lambda t: (t.created_at, t.ticket_id))
        ticket = pool[0]

        # Service this ticket
        service_duration = ticket.default_avg_min * rng.uniform(0.5, 1.8)
        # Occasionally a "difficult customer" — long service
        if rng.random() < 0.03:
            service_duration *= rng.uniform(2.0, 4.0)

        ticket.serving_started_at = now  # counter picks the ticket = CalledAt (target end point)
        ticket.service_end_at = now + service_duration  # record actual end time
        earliest_counter.free_at = now + service_duration
        earliest_counter.currently_serving = ticket

        # Track completed duration for rolling avg (with completion time for temporal filtering)
        completed_durations[ticket.prefix].append((now + service_duration, service_duration))

    # ── Build snapshot rows for each served ticket ──
    # Snapshot at queue_eligible_at (when the person starts waiting)
    rows: list[dict] = []
    served_tickets = [t for t in tickets if t.serving_started_at is not None]

    for t in served_tickets:
        snap_time = t.queue_eligible_at
        hour = int(snap_time // 60)
        if hour < 0 or hour > 23:
            hour = max(0, min(23, hour))

        is_peak = 1 if (9 <= hour <= 11 or 14 <= hour <= 16) else 0

        # Queue state at snapshot time
        same_svc_waiting = [
            o for o in tickets
            if o.service_code == t.service_code
            and not o.abandoned
            and o.queue_eligible_at <= snap_time
            and (o.serving_started_at is None or o.serving_started_at > snap_time)
        ]
        queue_length = len(same_svc_waiting)

        # NowServing: tickets being served at snapshot time for this service.
        # Uses actual service_end_at (observable: counter is busy) instead of
        # default_avg_min * 1.2 estimate which would be a data leak.
        now_serving = sum(
            1 for o in tickets
            if o.service_code == t.service_code
            and o.serving_started_at is not None
            and o.service_end_at is not None
            and o.serving_started_at <= snap_time
            and o.service_end_at > snap_time
        )

        # Counter counts
        active_counters = len(counters)
        svc_eligible_counters = sum(
            1 for c in counters if t.prefix in c.allowed_prefixes
        )

        # Rolling avg at snapshot time (only completions before snap_time — no future leakage)
        roll_avg = rolling_avg(t.prefix, before=snap_time)

        # Slot features
        slot_duration = slot_dur
        minutes_since_slot_start = None
        minutes_until_slot_end = None
        online_booked_in_slot = 0
        walkin_in_slot = 0
        slot_active = 0

        if t.slot_start is not None and t.slot_end is not None:
            minutes_since_slot_start = max(0.0, snap_time - t.slot_start)
            minutes_until_slot_end = max(0.0, t.slot_end - snap_time)
            slot_active = 1 if t.slot_start <= snap_time < t.slot_end else 0

            # Count others in same slot
            for o in tickets:
                if (o.service_code == t.service_code
                        and o.slot_start == t.slot_start
                        and o.slot_end == t.slot_end
                        and not o.abandoned):
                    if o.entry_type == 0:
                        online_booked_in_slot += 1
                    else:
                        walkin_in_slot += 1
        else:
            # Walk-in: slot features are NaN (HistGBR handles natively)
            slot_duration = slot_dur  # branch-level, always known
            minutes_since_slot_start = None
            minutes_until_slot_end = None

        # CallNextPriority: P0 if online + checked_in + slot active
        call_priority = 0 if (t.entry_type == 0 and t.checked_in and slot_active == 1) else 1

        # Heard lanes S: union of the prefixes of every counter that can serve my prefix (includes mine).
        # Fallback S = {my prefix} if no counter serves it (live CallNextQueueMath does the same).
        _, heard_prefixes = heard_lanes(t.prefix, counters)
        # Waiting in S at snap time: queue-eligible and not yet picked by a counter
        heard_waiting = [
            o for o in tickets
            if o.prefix in heard_prefixes
            and not o.abandoned
            and o.queue_eligible_at <= snap_time
            and (o.serving_started_at is None or o.serving_started_at > snap_time)
        ]
        cross_lane = len(heard_waiting)

        # PeopleAheadCallNext: per listening counter, eligible tickets in that counter's prefixes that it would
        # pick before me by (created_at, ticket_id); minimum over the counters (same-prefix only if no counter
        # serves my prefix). Same definition as live CallNextQueueMath. Each counter's prefixes are a subset of S,
        # so people_ahead <= cross_lane. (A union-of-S count overstated it, e.g. B001 kept 3 ahead after a
        # B-only counter opened.)
        people_ahead = people_ahead_call_next(t, tickets, counters, snap_time)

        # Target: CalledAt (counter pick) - QueueEligibleAt
        wait_min = t.serving_started_at - t.queue_eligible_at

        rows.append({
            # Queue state
            "QueueLength": queue_length,
            "CrossLaneQueueLength": cross_lane,
            "PeopleAheadCallNext": people_ahead,
            "NowServing": now_serving,
            # Capacity
            "ActiveCounters": active_counters,
            "ListeningCounters": svc_eligible_counters,
            # Time
            "HourOfDay": hour,
            "DayOfWeek": day_of_week,
            "IsPeakHour": is_peak,
            # Service duration
            "DefaultAvgServiceMinutes": t.default_avg_min,
            "RollingAvgServiceMinutes": round(roll_avg, 2),
            # Entry
            "EnqueueSequence": t.enqueue_seq,
            "EntryType": t.entry_type,
            "CheckedIn": 1 if t.checked_in else 0,
            # Slot features
            "SlotActive": slot_active,
            "CallNextPriority": call_priority,
            "SlotDurationMinutes": slot_duration,
            "EarlyCallMinutes": early_call,
            "MinutesUntilSlotEnd": minutes_until_slot_end,
            "MinutesSinceSlotStart": minutes_since_slot_start,
            "OnlineSlotsPerSlot": t.online_slots_per_slot,
            "OnlineBookedInSlot": online_booked_in_slot,
            "WalkInInSlot": walkin_in_slot,
            # Identity
            "BranchCode": t.branch_code,
            "ServiceCode": t.service_code,
            "TicketPrefix": t.prefix,
            # Target
            "WaitingMinutes": round(wait_min, 2),
        })

    # Also emit abandoned tickets (for training cleaning practice)
    for t in tickets:
        if t.abandoned:
            rows.append({
                "QueueLength": 0,
                "CrossLaneQueueLength": 0,
                "PeopleAheadCallNext": 0,
                "NowServing": 0,
                "ActiveCounters": len(counters),
                "ListeningCounters": sum(1 for c in counters if t.prefix in c.allowed_prefixes),
                "HourOfDay": int(t.created_at // 60),
                "DayOfWeek": day_of_week,
                "IsPeakHour": 0,
                "DefaultAvgServiceMinutes": t.default_avg_min,
                "RollingAvgServiceMinutes": round(rolling_avg(t.prefix, before=t.created_at), 2),
                "EnqueueSequence": t.enqueue_seq,
                "EntryType": t.entry_type,
                "CheckedIn": 0,
                "SlotActive": 0,
                "CallNextPriority": 1,
                "SlotDurationMinutes": slot_dur,
                "EarlyCallMinutes": early_call,
                "MinutesUntilSlotEnd": None,
                "MinutesSinceSlotStart": None,
                "OnlineSlotsPerSlot": t.online_slots_per_slot,
                "OnlineBookedInSlot": 0,
                "WalkInInSlot": 0,
                "BranchCode": t.branch_code,
                "ServiceCode": t.service_code,
                "TicketPrefix": t.prefix,
                "WaitingMinutes": None,  # abandoned — no service
            })

    return rows

print('Simulation function defined.')

In [ ]:
%%time
# Generate 30K rows across multiple days and branches
rng = np.random.default_rng(42)
all_rows = []
day_count = 0
TARGET_ROWS = 30000

while len(all_rows) < TARGET_ROWS:
    for br in BRANCHES:
        if len(all_rows) >= TARGET_ROWS: break
        dow = day_count % 5
        n_prof = min(len(PROFILES), max(2, int(rng.integers(3, 7))))
        all_rows.extend(simulate_day(rng, br, dow, n_prof))
    day_count += 1

df = pd.DataFrame(all_rows)

# Winsorize at p99
served = df['WaitingMinutes'].notna()
if served.sum() > 0:
    p99 = df.loc[served, 'WaitingMinutes'].quantile(0.99)
    df.loc[served, 'WaitingMinutes'] = df.loc[served, 'WaitingMinutes'].clip(upper=p99)

df['DataSource'] = 'synthetic'
print(f'Generated {len(df)} rows ({served.sum()} served)')
print(f'Wait time: mean={df["WaitingMinutes"].mean():.1f}, median={df["WaitingMinutes"].median():.1f}, p95={df["WaitingMinutes"].quantile(0.95):.1f}')

In [ ]:
# Visualize synthetic data distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

s = df[df['WaitingMinutes'].notna()]
axes[0].hist(s['WaitingMinutes'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Wait Time (min)')
axes[0].set_title(f'Wait Time Distribution (n={len(s):,})')
axes[0].axvline(s['WaitingMinutes'].median(), color='red', ls='--', label=f'median={s["WaitingMinutes"].median():.1f}')
axes[0].legend()

axes[1].bar(range(9,17), [s[s['HourOfDay']==h]['WaitingMinutes'].mean() for h in range(9,17)], color='coral')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Avg Wait (min)')
axes[1].set_title('Wait Time by Hour')

svc_avg = s.groupby('ServiceCode')['WaitingMinutes'].mean().sort_values(ascending=False)
axes[2].barh(svc_avg.index, svc_avg.values, color='mediumseagreen')
axes[2].set_xlabel('Avg Wait (min)')
axes[2].set_title('Wait Time by Service')

plt.tight_layout()
plt.show()

### 1b. Combine External Datasets (Optional)

Upload `hospital_engineered.csv` and `college_engineered.csv` to Colab, then run the cell below.
If you skip this cell, the pipeline trains on **synthetic data only** (recommended for launch).

Three safeguards prevent external data from hurting the model:
1. **Filter**: Hospital keeps only `active_counters ≥ 3` (drops 1/2-counter disaster scenarios)
2. **Scale normalization**: Align external median wait to synthetic median
3. **Sample weighting**: External gets weight 0.1 vs synthetic 1.0 during training

In [ ]:
# ── Upload external datasets (Colab only) ──
# Option 1: Drag files into the left sidebar Files panel (easiest)
# Option 2: Run this cell — it will only upload files that are missing

import os

needed = ['hospital_engineered.csv', 'college_engineered.csv']
missing = [f for f in needed if not os.path.exists(f)]

if missing:
    try:
        from google.colab import files
        print(f'Missing: {missing} — uploading...')
        for f in missing:
            print(f'  Select {f}:')
            uploaded = files.upload()
    except ImportError:
        print('Not in Colab — place CSV files in working directory.')
else:
    print('Both files found (from sidebar upload). Ready for next cell.')

In [ ]:
# ── Combine External Datasets with Synthetic Data ──
# Skip this cell if you only want to train on synthetic data.

import os
TARGET = 'WaitingMinutes'
external_frames = []

# --- Hospital Queue ---
hospital_path = 'hospital_engineered.csv'  # Upload to Colab first
if os.path.exists(hospital_path):
    h = pd.read_csv(hospital_path)
    h = h.rename(columns={
        'queue_length': 'QueueLength', 'people_ahead': 'PeopleAheadCallNext',
        'now_serving': 'NowServing', 'active_counters': 'ActiveCounters',
        'hour': 'HourOfDay', 'is_peak': 'IsPeakHour',
        'priority': 'CallNextPriority', 'default_avg_service_min': 'DefaultAvgServiceMinutes',
        'wait_min': TARGET,
    })
    # Safeguard 1: Filter — drop 1/2-counter disaster scenarios
    n_before = len(h)
    h = h[h['ActiveCounters'] >= 3].copy()
    h['CrossLaneQueueLength'] = h['QueueLength']
    h['ListeningCounters'] = h['ActiveCounters']
    h['BranchCode'] = 900001
    h['ServiceCode'] = 'HOSPITAL'
    h['TicketPrefix'] = 'H'
    h['EntryType'] = 1
    h['CheckedIn'] = 1
    h['EnqueueSequence'] = range(1, len(h) + 1)
    h['DayOfWeek'] = 2
    h['DataSource'] = 'hospital'
    external_frames.append(h)
    print(f'Hospital: {len(h)} rows (filtered from {n_before}, kept counters>=3)')
else:
    print(f'Hospital file not found ({hospital_path}) — skipped')

# --- College Admissions ---
college_path = 'college_engineered.csv'  # Upload to Colab first
if os.path.exists(college_path):
    c = pd.read_csv(college_path)
    c = c.rename(columns={
        'queue_length': 'QueueLength', 'people_ahead': 'PeopleAheadCallNext',
        'now_serving': 'NowServing', 'active_counters': 'ActiveCounters',
        'hour': 'HourOfDay', 'day_of_week': 'DayOfWeek',
        'is_peak': 'IsPeakHour', 'avg_service_min': 'DefaultAvgServiceMinutes',
        'wait_min': TARGET,
    })
    c['CrossLaneQueueLength'] = c['QueueLength']
    c['ListeningCounters'] = c['ActiveCounters']
    c['BranchCode'] = 900002
    c['ServiceCode'] = 'COLLEGE'
    c['TicketPrefix'] = 'C'
    c['EntryType'] = 1
    c['CheckedIn'] = 1
    c['EnqueueSequence'] = range(1, len(c) + 1)
    c['DataSource'] = 'college'
    external_frames.append(c)
    print(f'College: {len(c)} rows')
else:
    print(f'College file not found ({college_path}) — skipped')

# --- Merge ---
if external_frames:
    ext_df = pd.concat(external_frames, ignore_index=True)

    # Safeguard 2: Scale normalization — align median to synthetic
    synthetic_median = df[TARGET].dropna().median()
    for src in ext_df['DataSource'].unique():
        mask = ext_df['DataSource'] == src
        src_median = ext_df.loc[mask, TARGET].median()
        if src_median > 0:
            scale = synthetic_median / src_median
            ext_df.loc[mask, TARGET] *= scale
            ext_df['DefaultAvgServiceMinutes'] = ext_df['DefaultAvgServiceMinutes'].astype(float)
            ext_df.loc[mask, 'DefaultAvgServiceMinutes'] *= scale
            print(f'Scaled {src}: median {src_median:.1f} -> {ext_df.loc[mask, TARGET].median():.1f} min (x{scale:.2f})')

    # Cap external at 50% of synthetic (prevent domination)
    max_ext = int(len(df) * 0.5)
    if len(ext_df) > max_ext:
        ext_df = ext_df.sample(n=max_ext, random_state=42)
        print(f'Downsampled external to {max_ext} rows (50% of synthetic)')

    df = pd.concat([df, ext_df], ignore_index=True)
    print(f'\nCombined total: {len(df)} rows')
    for src in df['DataSource'].unique():
        print(f'  {src}: {(df["DataSource"] == src).sum()}')
else:
    print('No external datasets found — training on synthetic only')

## 2. Data Cleansing

In [ ]:
TARGET = 'WaitingMinutes'
n_start = len(df)

# Drop abandoned tickets (no service = no target)
df = df.dropna(subset=[TARGET])

# Safety net: drop negative waits (system glitches). The simulation models online check-in time,
# so QueueEligibleAt <= CalledAt always holds and this removes ~nothing. Keep 0 = instant service.
df = df[df[TARGET] >= 0].copy()

# Winsorize per source
for src in df['DataSource'].unique():
    mask = df['DataSource'] == src
    p99 = df.loc[mask, TARGET].quantile(0.99)
    p01 = df.loc[mask, TARGET].quantile(0.01)
    df.loc[mask, TARGET] = df.loc[mask, TARGET].clip(lower=p01, upper=p99)

# Dedup
df = df.drop_duplicates()

print(f'Cleaning: {n_start} -> {len(df)} rows (dropped {n_start - len(df)}, {(n_start-len(df))/n_start:.1%})')
print(f'Target range: [{df[TARGET].min():.1f}, {df[TARGET].max():.1f}] min')
print(f'Mean={df[TARGET].mean():.1f}, Median={df[TARGET].median():.1f}')

## 3. Feature Engineering

**26 features** aligned with production `MlHttpWaitTimeEstimator.BuildPayload`:
- Queue state: QueueLength, CrossLaneQueueLength, PeopleAheadCallNext, NowServing
- Capacity: ActiveCounters, ListeningCounters
- Time: HourOfDay, DayOfWeek, IsPeakHour
- Service: DefaultAvgServiceMinutes, RollingAvgServiceMinutes
- Entry: EnqueueSequence, EntryType, CheckedIn
- Slot (NaN for walk-ins): SlotActive, CallNextPriority, SlotDurationMinutes, MinutesUntilSlotEnd, MinutesSinceSlotStart, OnlineSlotsPerSlot, OnlineBookedInSlot, WalkInInSlot
- Branch policy: EarlyCallMinutes (= Branch.OnlineEarlyCallMinutes, set by manager 0..30; NaN for external datasets)
- Identity: BranchCode, ServiceCode, TicketPrefix

In [ ]:
FEATURE_NUMERIC = [
    'QueueLength', 'CrossLaneQueueLength', 'PeopleAheadCallNext', 'NowServing',
    'ActiveCounters', 'ListeningCounters',
    'HourOfDay', 'DayOfWeek', 'IsPeakHour',
    'DefaultAvgServiceMinutes',
    'RollingAvgServiceMinutes',
    'EnqueueSequence', 'EntryType', 'CheckedIn',
    'SlotActive', 'CallNextPriority', 'SlotDurationMinutes', 'EarlyCallMinutes',
    'MinutesUntilSlotEnd', 'MinutesSinceSlotStart',
    'OnlineSlotsPerSlot', 'OnlineBookedInSlot', 'WalkInInSlot',
    'BranchCode',
]
FEATURE_CAT = ['ServiceCode', 'TicketPrefix']
ALL_FEATURES = FEATURE_NUMERIC + FEATURE_CAT

# Ensure columns exist
for col in FEATURE_NUMERIC:
    if col not in df.columns: df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors='coerce')
for col in FEATURE_CAT:
    if col not in df.columns: df[col] = 'UNKNOWN'

# Report NaN (expected for slot features on walk-ins, RollingAvg on external data)
nan_counts = df[FEATURE_NUMERIC].isna().sum()
nan_feats = nan_counts[nan_counts > 0]
print(f'{len(ALL_FEATURES)} features ({len(FEATURE_NUMERIC)} numeric + {len(FEATURE_CAT)} categorical)')
if len(nan_feats) > 0:
    print('\nNaN features (handled natively by HistGBR):')
    for f, c in nan_feats.items():
        print(f'  {f}: {c} ({c/len(df):.1%})')

## 4. Model Training

- **HistGradientBoostingRegressor** with **quantile loss (p75)** — asymmetric: under-prediction hurts UX more than over-prediction. Predicting p75 means 75% of customers are served within the estimated time.
- **Temporal split** (train on earlier data, test on later) to prevent data leakage
- **No imputation needed** — HistGBR handles NaN natively for slot features

In [ ]:
# Per-source temporal split (same as train_wait_model.py)
# Each DataSource is split independently to prevent all external data
# ending up on one side (this bug caused MAE=468 before we fixed it).

sort_col = 'EnqueueSequence'
test_ratio = 0.2

if 'DataSource' not in df.columns:
    df['DataSource'] = 'synthetic'

train_parts, test_parts = [], []
for source, group in df.groupby('DataSource'):
    group = group.sort_values(sort_col)
    split_idx = int(len(group) * (1 - test_ratio))
    train_parts.append(group.iloc[:split_idx])
    test_parts.append(group.iloc[split_idx:])

train_df = pd.concat(train_parts, ignore_index=True)
test_df  = pd.concat(test_parts, ignore_index=True)

X_train, y_train = train_df[ALL_FEATURES], train_df[TARGET].astype(float)
X_test,  y_test  = test_df[ALL_FEATURES],  test_df[TARGET].astype(float)

print(f'Per-source temporal split: train={len(train_df)}, test={len(test_df)} ({test_ratio:.0%} test)')
for src in df['DataSource'].unique():
    n_tr = (train_df['DataSource'] == src).sum()
    n_te = (test_df['DataSource'] == src).sum()
    print(f'  {src}: train={n_tr}, test={n_te}')
print(f'Train wait: mean={y_train.mean():.1f}, Test wait: mean={y_test.mean():.1f}')

In [ ]:
%%time
# Build and train model
preprocessor = ColumnTransformer([
    ('num', 'passthrough', FEATURE_NUMERIC),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), FEATURE_CAT),
])

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', HistGradientBoostingRegressor(
        loss='quantile',       # asymmetric: under-prediction hurts UX more than over-prediction
        quantile=0.75,         # predict p75 — 75% of customers served within this estimate
        max_depth=6, learning_rate=0.08, max_iter=300,
        min_samples_leaf=8, random_state=42,
    )),
])

# Safeguard 3: Sample weighting — synthetic/primary=1.0, external=0.1
# This ensures external data adds signal without dominating the loss function.
train_weights = train_df['DataSource'].map(
    lambda s: 1.0 if s in ('synthetic', 'primary', 'sql') else 0.1
).values
pipeline.fit(X_train, y_train, model__sample_weight=train_weights)

weight_summary = train_df['DataSource'].value_counts()
print('Model trained with quantile loss (p75) and sample weighting:')
print('  p75 = 75% of customers served within the predicted time (conservative estimate)')
for src in weight_summary.index:
    w = 1.0 if src in ('synthetic', 'primary', 'sql') else 0.1
    print(f'  {src}: {weight_summary[src]} rows x weight {w}')

## 5. Evaluation

In [ ]:
# ML predictions
ml_pred = np.maximum(0, pipeline.predict(X_test))

# Formula baseline (matches production WaitTimeEstimator.cs)
def formula_baseline(row):
    ahead = max(0, row.get('PeopleAheadCallNext', 0) or 0)
    now_s = max(0, row.get('NowServing', 0) or 0)
    avg = row.get('DefaultAvgServiceMinutes', 10) or 10
    counters = max(1, row.get('ListeningCounters', 1) or 1)
    return max(0, (ahead + now_s) * avg / counters)

fm_pred = test_df.apply(formula_baseline, axis=1).values

# Metrics
ml_mae  = mean_absolute_error(y_test, ml_pred)
ml_rmse = np.sqrt(mean_squared_error(y_test, ml_pred))
ml_r2   = r2_score(y_test, ml_pred)
fm_mae  = mean_absolute_error(y_test, fm_pred)
fm_rmse = np.sqrt(mean_squared_error(y_test, fm_pred))
fm_r2   = r2_score(y_test, fm_pred)

# Cross-validation (on full dataset)
# FIX: Sort by DataSource → BranchCode → EnqueueSequence so TimeSeriesSplit
# respects temporal order within each branch. Previously sorted only by
# EnqueueSequence, which is independently numbered per branch/source —
# mixing them destroyed the physical time ordering.
cv_sort_cols = ['DataSource', 'BranchCode', 'EnqueueSequence']
X_all = df.sort_values(cv_sort_cols)[ALL_FEATURES]
y_all = df.sort_values(cv_sort_cols)[TARGET].astype(float)
tscv = TimeSeriesSplit(n_splits=5)
cv_mae = -cross_val_score(pipeline, X_all, y_all, cv=tscv, scoring='neg_mean_absolute_error')

print('=' * 60)
print('  RESULTS')
print('=' * 60)
print(f'  ML Model:     MAE = {ml_mae:.2f} min   RMSE = {ml_rmse:.2f}   R2 = {ml_r2:.4f}')
print(f'  Formula:      MAE = {fm_mae:.2f} min   RMSE = {fm_rmse:.2f}   R2 = {fm_r2:.4f}')
print(f'  CV MAE:       {np.mean(cv_mae):.2f} min (5-fold TimeSeriesSplit)')
print(f'  Improvement:  ML is {fm_mae/ml_mae:.1f}x more accurate than formula')

In [ ]:
# Feature Importance
from sklearn.inspection import permutation_importance

perm = permutation_importance(pipeline, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
imp = sorted(zip(ALL_FEATURES, perm.importances_mean), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(10, 6))
names, vals = zip(*imp[:15])
ax.barh(range(len(names)), vals, color='steelblue')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel('Permutation Importance')
ax.set_title('Top 15 Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual + Residuals
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Predicted vs Actual
ax = axes[0]
ax.scatter(y_test, ml_pred, s=4, alpha=0.3, color='steelblue', label='ML')
ax.scatter(y_test, fm_pred, s=4, alpha=0.15, color='orange', label='Formula')
lim = [0, max(y_test.max(), ml_pred.max()) * 1.05]
ax.plot(lim, lim, 'r--', lw=1, label='Perfect')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Actual (min)'); ax.set_ylabel('Predicted (min)')
ax.set_title('Predicted vs Actual')
ax.legend(markerscale=4)

# Residual histogram
ax2 = axes[1]
ml_resid = ml_pred - y_test.values
fm_resid = fm_pred - y_test.values
rng_val = max(abs(ml_resid).max(), 30)
ax2.hist(ml_resid, bins=50, range=(-rng_val, rng_val), alpha=0.6, color='steelblue', label=f'ML (MAE={ml_mae:.1f})')
ax2.hist(fm_resid, bins=50, range=(-rng_val, rng_val), alpha=0.4, color='orange', label=f'Formula (MAE={fm_mae:.1f})')
ax2.axvline(0, color='red', ls='--')
ax2.set_xlabel('Residual (min)'); ax2.set_title('Residual Distribution')
ax2.legend()

# Error by service type
ax3 = axes[2]
test_df_copy = test_df.copy()
test_df_copy['ml_error'] = np.abs(ml_pred - y_test.values)
test_df_copy['fm_error'] = np.abs(fm_pred - y_test.values)
svc_ml = test_df_copy.groupby('ServiceCode')['ml_error'].mean().sort_values(ascending=False)
svc_fm = test_df_copy.groupby('ServiceCode')['fm_error'].mean()
x_pos = range(len(svc_ml))
ax3.bar([p - 0.2 for p in x_pos], svc_ml.values, 0.4, color='steelblue', label='ML')
ax3.bar([p + 0.2 for p in x_pos], [svc_fm.get(s, 0) for s in svc_ml.index], 0.4, color='orange', label='Formula')
ax3.set_xticks(list(x_pos))
ax3.set_xticklabels(svc_ml.index, rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('MAE (min)'); ax3.set_title('MAE by Service Type')
ax3.legend()

plt.suptitle('QMS Wait-Time Model Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

| Metric | ML Model | Formula Baseline |
|--------|----------|------------------|
| MAE    | ~11-13 min | ~20-22 min |
| R²     | ~0.60-0.64 | < 0 (worse than mean) |
| Improvement | **1.7-1.8x** more accurate | baseline |

### Architecture
```
Customer takes ticket
    → WaitTimeFeatureBuilder computes 26 features
    → MlHttpWaitTimeEstimator sends to Python sidecar
    → HistGradientBoosting predicts wait time
    → Sanity clamp (50% formula ≤ ML ≤ 150% simulation)
    → Display on ticket / app
```

Synthetic data models online check-in time, so online tickets only become eligible after the customer checks in (no negative waits).

### Deployment Phases
1. **Pre-launch**: Train on synthetic data (this notebook)
2. **Post-launch**: Retrain on real `ML_TRAINING_DATA` from system
3. **Mature**: Full real data, consider Two-Stage model for anomalies

In [ ]:
import joblib, json
from pathlib import Path

MODEL_PATH = Path('wait_model.joblib')
METRICS_PATH = Path('model_metrics.json')

# ── Export model bundle (same format as train_wait_model.py) ──
joblib.dump({
    'pipeline': pipeline,
    'feature_numeric': FEATURE_NUMERIC,
    'feature_categorical': FEATURE_CAT,
    'target': TARGET,
    'version': 'v2',
}, MODEL_PATH)

# ── Export metrics ──
metrics = {
    'data_source': 'notebook (synthetic)',
    'row_count': len(df),
    'test_ml': {'mae_minutes': round(ml_mae, 3), 'rmse_minutes': round(ml_rmse, 3), 'r2': round(ml_r2, 4)},
    'test_formula_baseline': {'mae_minutes': round(fm_mae, 3), 'rmse_minutes': round(fm_rmse, 3), 'r2': round(fm_r2, 4)},
    'cv_mae_mean': round(float(np.mean(cv_mae)), 3),
    'feature_count': len(ALL_FEATURES),
    'features_numeric': FEATURE_NUMERIC,
    'features_categorical': FEATURE_CAT,
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2))

print(f'Saved: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024:.0f} KB)')
print(f'Saved: {METRICS_PATH}')
print(f'\nDeploy: copy wait_model.joblib to ml/ then restart serve_wait_model.py')